In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

data_batch_1 = unpickle("./data/CIFAR-10/data_batch_1")
data_batch_2 = unpickle("./data/CIFAR-10/data_batch_2")
data_batch_3 = unpickle("./data/CIFAR-10/data_batch_3")
data_batch_4 = unpickle("./data/CIFAR-10/data_batch_4")
data_batch_5 = unpickle("./data/CIFAR-10/data_batch_5")
test_batch = unpickle("./data/CIFAR-10/test_batch")
x_train = np.concatenate((data_batch_1[b'data'], data_batch_2[b'data'], data_batch_3[b'data'], data_batch_4[b'data'], data_batch_5[b'data']), axis=0)
y_train = np.concatenate((data_batch_1[b'labels'], data_batch_2[b'labels'], data_batch_3[b'labels'], data_batch_4[b'labels'], data_batch_5[b'labels']), axis=0)
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])
labels_names = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]

x_train = x_train.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
x_test = x_test.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0

mean = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32).reshape(1, 3, 1, 1)
std = np.array([0.2470, 0.2435, 0.2616], dtype=np.float32).reshape(1, 3, 1, 1)

x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

y_train = np.array(y_train)
y_test = np.array(y_test)

x_train_tensor = torch.from_numpy(x_train)
y_train_tensor = torch.from_numpy(y_train).long()

x_test_tensor = torch.from_numpy(x_test)
y_test_tensor = torch.from_numpy(y_test).long()

C:\Users\thoma\AppData\Local\Temp\ipykernel_17972\4092496055.py:3: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dict = pickle.load(fo, encoding='bytes')


In [3]:
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

trainloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [4]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.maxpool1 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.maxpool2 = nn.MaxPool2d(2, 2)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.flatten = nn.Flatten()
        self.n = nn.Linear(8*8*64, 10)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.maxpool1(x)

        x = F.relu(self.conv3(x))
        x = self.maxpool2(x)

        x = F.relu(self.conv4(x))

        x = self.flatten(x)
        x = self.n(x)
        
        return x

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), 
                      lr=0.01,
                      momentum=0.9,
                      weight_decay=5e-4)

In [7]:
def train(model, trainloader, testloader, epochs=10):
    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        running_correct = 0
        running_total = 0

        pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            running_correct += (preds == labels).sum().item()
            running_total += labels.size(0)

            avg_loss = running_loss / running_total
            avg_acc = running_correct / running_total

            pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")

        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in testloader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                preds = outputs.argmax(dim=1)

                test_loss += loss.item() * images.size(0)
                test_correct += (preds == labels).sum().item()
                test_total += labels.size(0)

        print(
            f"Test - loss: {test_loss / test_total:.4f} | "
            f"acc: {test_correct / test_total:.4f}"
        )

In [9]:
train(model, trainloader, testloader, epochs=10)

Epoch 1/10: 100%|██████████| 782/782 [02:11<00:00,  5.97it/s, acc=0.4550, loss=1.5039]


Test - loss: 1.1388 | acc: 0.5954


Epoch 2/10: 100%|██████████| 782/782 [02:10<00:00,  6.00it/s, acc=0.6592, loss=0.9748]


Test - loss: 0.8494 | acc: 0.7039


Epoch 3/10: 100%|██████████| 782/782 [02:14<00:00,  5.80it/s, acc=0.7278, loss=0.7786]


...

Test - loss: 0.7253 | acc: 0.7648


Epoch 9/10: 100%|██████████| 782/782 [02:09<00:00,  6.05it/s, acc=0.8726, loss=0.3649]


Test - loss: 0.7438 | acc: 0.7627


Epoch 10/10: 100%|██████████| 782/782 [02:08<00:00,  6.09it/s, acc=0.8844, loss=0.3269]


Test - loss: 0.8118 | acc: 0.7590


In [ ]:
def predict(model, x):
    model.eval()
    with torch.no_grad():
        x = x.to(device)
        outputs = model(x)
        preds = outputs.argmax(dim=1)
    return preds

In [11]:
def predict_topk(model, x, class_names, k=3):
    model.eval()
    with torch.no_grad():
        x = x.to(device)
        outputs = model(x)
        probs = F.softmax(outputs, dim=1)
        top_probs, top_idxs = torch.topk(probs, k=k, dim=1)

    for i in range(x.size(0)):
        print(f"Image {i}:")
        for p, idx in zip(top_probs[i], top_idxs[i]):
            print(f"  {class_names[idx.item()]} : {p.item()*100:.2f}%")

In [ ]:
k = 17

image, target = test_dataset[k]
predict_topk(model, image.unsqueeze(0), class_names=labels_names)
print(labels_names[target])

Image 0:
  cheval : 98.22%
  chat : 1.26%
  chien : 0.45%
cheval


In [36]:
print(labels_names[predict(model, image.unsqueeze(0))])

cheval
